# Batch correction with scVI

scVI (Lopez et al., *Nat Methods* 2018) is the canonical deep variational autoencoder for scRNA-seq. The decoder is conditioned on batch, so the latent representation strips batch effect while preserving cell-type identity. Best for atlases with many batches and uneven composition.

This is one of the **omicverse batch-correction zoo** tutorials. For an overview of all backends, the recommendation tree, and the unified `_BATCH_OBSM` schema, see [batch/index](../index.md). For the side-by-side comparison of all backends on the NeurIPS 2021 multi-batch benchmark, see [t_single_batch](../t_single_batch.ipynb).

Optional dependency: `pip install scvi-tools` or `conda install scvi-tools -c conda-forge`.

**Input contract**: raw integer counts must live in `adata.layers['counts']`. The wrapper passes `layer='counts'` to `setup_anndata` automatically.


## Load a multi-batch dataset

We use the same toy multi-batch AnnData as the other zoo tutorials — three NeurIPS 2021 batches concatenated. Replace with your own dataset by changing `adata` and `batch_key` below.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

# Replace these URLs with your own dataset; the three are
# the same multi-batch dataset used by t_single_batch.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.obs['batch'] = adata.obs['batch'].astype('category')
adata

## Preprocess + PCA (shared across all backends)

Every backend in the zoo starts from the same QC'd, log-normalised AnnData with `scaled|original|X_pca` in obsm. Read [t_single_batch](../t_single_batch.ipynb) for the full discussion of these steps.

In [ ]:
adata = ov.pp.qc(adata,
                 tresh={'mito_perc': 0.2, 'nUMIs': 500,
                        'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson',
                         n_HVGs=2000, batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## Run `ov.single.batch_correction(methods='scVI')`

The wrapper routes method-specific kwargs to the right destination — for scvi-tools backends this includes splitting between `__init__` (architecture) and `.train()` (optimisation). See the **Key parameters** section below.

In [ ]:
model = ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='scVI',
    # Architecture (SCVI.__init__):
    n_hidden=128, n_latent=20, n_layers=2, dropout_rate=0.1,
    # Optimisation (SCVI.train) — routed by the kwarg splitter:
    max_epochs=200, batch_size=128, early_stopping=True,
)
model

## Visualise the corrected embedding

Every backend writes its corrected representation to a stable obsm key — for **scVI** it is `adata.obsm['X_scVI']`. We project it via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_scvi'] = ov.utils.mde(adata.obsm['X_scVI'])
ov.pl.embedding(
    adata,
    basis='X_mde_scvi',
    color=['batch'],
    frameon='small',
    title='scVI — coloured by batch',
)

## Key parameters

**Architecture** (routed to `scvi.model.SCVI.__init__`):
- `n_latent` — latent dimension (default 10).
- `n_hidden` — hidden layer width (default 128).
- `n_layers` — number of hidden layers (default 1).
- `dropout_rate` — dropout (default 0.1).
- `gene_likelihood` — one of `'zinb'` / `'nb'` / `'poisson'`.

**Optimisation** (routed to `scvi.model.SCVI.train`):
- `max_epochs`, `batch_size`, `early_stopping`, `accelerator`, `devices`, `train_size`, `validation_size`.

The kwarg splitter in `ov.single.batch_correction` partitions your `**kwargs` between these two destinations automatically.


## Related tutorials

- `scANVI` — semi-supervised extension when you have partial labels.
- `totalVI` — joint RNA + protein for CITE-seq data.
- `harmony` — non-DL alternative when scVI training time is prohibitive.

For a side-by-side comparison of every backend on the same benchmark + scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).